In [ ]:
import os
from pathlib import Path
import pandas as pd
from ultralytics import YOLO
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

PATH_KGO_FULL = '../pipeline/images/kgo_full'   
PATH_KGO_EMPTY = '../pipeline/images/kgo_empty' 


MODEL_PATH = 'runs/detect/runs/train/kgo_multitask/weights/best.pt'

CONFIDENCE_THRESHOLD = 0.25

print(f"Full folder: {PATH_KGO_FULL}")
print(f"Empty folder: {PATH_KGO_EMPTY}")

Full folder: ../pipeline/images/kgo_full
Empty folder: ../pipeline/images/kgo_empty


In [ ]:
model = YOLO(MODEL_PATH)

Модель загружена.


In [ ]:
def predict_kgo(image_path, conf_thres=CONFIDENCE_THRESHOLD):
    results = model(image_path, conf=conf_thres, verbose=False)
    kgo_dets = []
    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            if cls in (1, 2):
                kgo_dets.append((cls, box.conf[0].item()))
    if not kgo_dets:
        return ""
    best_cls, _ = max(kgo_dets, key=lambda x: x[1])
    return 'kgo_empty' if best_cls == 1 else 'kgo_full'

In [ ]:
records = []

def process_folder(folder, ground_truth):
    folder_path = Path(folder)
    if not folder_path.exists():
        print(f"Папка {folder_path} не найдена!")
        return
    image_files = list(folder_path.glob('*.*'))
    image_files = [f for f in image_files if f.suffix.lower() in ('.jpg','.jpeg','.png')]
    for img_path in tqdm(image_files, desc=f'Папка {folder_path.name}'):
        pred = predict_kgo(str(img_path))
        records.append({
            'image': img_path.name,
            'ground_truth': ground_truth,
            'prediction': pred,
            'correct': ground_truth == pred
        })

process_folder(PATH_KGO_FULL, 'kgo_full')
process_folder(PATH_KGO_EMPTY, 'kgo_empty')

df = pd.DataFrame(records)
df.head()

Обрабатываем 463 изображений из ..\pipeline\images\kgo_full


Папка kgo_full:   0%|          | 0/463 [00:00<?, ?it/s]

Обрабатываем 161 изображений из ..\pipeline\images\kgo_empty


Папка kgo_empty:   0%|          | 0/161 [00:00<?, ?it/s]

Всего обработано: 624


,image,ground_truth,prediction,correct
0,01.04-11.04.20250203706693425940c91fc142716e23...,kgo_full,kgo_full,True
1,01.04-11.04.202502ef714faf128529afb7a7edfadc1f...,kgo_full,kgo_full,True
2,01.04-11.04.2025074e52e45336d8cf36fbec26702e1d...,kgo_full,kgo_full,True
3,01.04-11.04.2025089fcd89f91d27f2952440a3f5d763...,kgo_full,kgo_full,True
4,01.04-11.04.20250915a5be85a125d4ef1847f4b862df...,kgo_full,kgo_full,True


In [ ]:

accuracy = df['correct'].mean()
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
labels = ['kgo_empty', 'kgo_full', '']
print(classification_report(df['ground_truth'], df['prediction'], labels=labels, zero_division=0))

Accuracy: 0.5978

Classification Report:
              precision    recall  f1-score   support

   kgo_empty       0.00      0.00      0.00       161
    kgo_full       0.71      0.81      0.75       463
                   0.00      0.00      0.00         0

    accuracy                           0.60       624
   macro avg       0.24      0.27      0.25       624
weighted avg       0.52      0.60      0.56       624

